# 手写数字分类

比较 SVM 和 KNN。实验函数在同目录的 `experiment.py` 中。

## 数据

每张 8×8 图片展平成 64 个特征。`y` 保存对应的数字标签。

In [1]:
from pathlib import Path
import numpy as np
from IPython.display import Image, display
from experiment import load_split, choose_k, make_models, evaluate

digits, X, y, train, test = load_split()
print("图片形状:", digits.images.shape)
print("特征形状:", X.shape)
print("标签形状:", y.shape)
print("第一张图片的标签:", y[0])
print(digits.images[0])

图片形状: (1797, 8, 8)
特征形状: (1797, 64)
标签形状: (1797,)
第一张图片的标签: 0
[[ 0.  0.  5. 13.  9.  1.  0.  0.]
 [ 0.  0. 13. 15. 10. 15.  5.  0.]
 [ 0.  3. 15.  2.  0. 11.  8.  0.]
 [ 0.  4. 12.  0.  0.  8.  8.  0.]
 [ 0.  5.  8.  0.  0.  9.  8.  0.]
 [ 0.  4. 11.  0.  1. 12.  7.  0.]
 [ 0.  2. 14.  5. 10. 12.  0.  0.]
 [ 0.  0.  6. 13. 10.  0.  0.  0.]]


## 划分训练集和测试集

保持类别比例，测试集占 25%。训练和测试样本没有重叠。

In [2]:
assert set(train).isdisjoint(test)
print("训练样本:", len(train))
print("测试样本:", len(test))

训练样本: 1347
测试样本: 450


## 选择 KNN 参数

只用训练集比较五个 k。每个分数是五折验证的平均准确率；测试集不参与选择。

In [3]:
k = choose_k(X[train], y[train])
print("选择的 k:", k)

k=1: CV accuracy=0.9681
k=3: CV accuracy=0.9733
k=5: CV accuracy=0.9748
k=7: CV accuracy=0.9666
k=9: CV accuracy=0.9636
选择的 k: 5


## 训练和评估

模型在训练集上 `fit`，在测试集上 `predict`。下面的准确率来自固定测试集。运行会更新 `results/`。

In [4]:
models = make_models(k)
metrics = evaluate(models, digits, X, y, train, test, Path("results"))
for row in metrics:
    assert row["errors"] + round(row["accuracy"] * len(test)) == len(test)

svm: accuracy=0.9800, errors=9
knn: accuracy=0.9644, errors=16


## 错误图片

混淆矩阵的行是真实标签，列是预测标签。错误图片显示真实与预测标签，id 是原始数据编号。

![SVM 混淆矩阵](results/svm_confusion_matrix.png)

![SVM 前八个错误](results/svm_errors.png)

In [5]:
display(Image(filename="results/svm_errors.png"))
# 第一张错误图片编号为 129，可以换成图中其他编号。
idx = 129
print("原始标签:", y[idx])
print(digits.images[idx])

原始标签: 8
[[ 0.  0.  7. 13. 15.  5.  0.  0.]
 [ 0.  0.  8. 16. 16. 12.  0.  0.]
 [ 0.  0.  7. 16. 15.  3.  0.  0.]
 [ 0.  0.  6. 16.  5.  0.  0.  0.]
 [ 0.  0.  5. 16.  2.  0.  0.  0.]
 [ 0.  0.  8. 16.  6.  0.  0.  0.]
 [ 0.  0. 12. 12. 13.  0.  0.  0.]
 [ 0.  0.  5. 13. 10.  0.  0.  0.]]
